# CDS524 Assignment 1: Q-Learning Analysis

## Reinforcement Learning Game Design

This notebook analyzes the Q-learning training data from the Horror Survival Game AI.

### Project Overview
- **Algorithm**: Q-Learning with Epsilon-Greedy Exploration
- **State Space**: 288 discrete states (distance × angle × health × attack × detection)
- **Action Space**: 6 actions (move forward/back/left/right, attack, idle)
- **Game Engine**: Godot 4.6

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install matplotlib numpy --quiet

# Import libraries
import json
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import os

print("✅ Setup complete!")

## 2. Upload Training Data

Upload your training files:
1. `training_stats.json` - Training statistics
2. `training_save_qtable.json` - Q-table data (optional)

In [ ]:
# Upload training data files
print("📤 Please upload your training_stats.json file:")
uploaded_stats = files.upload()

print("\n📤 Please upload your Q-table file (optional):")
print("Press Cancel if you don't have it")
try:
    uploaded_qtable = files.upload()
except:
    uploaded_qtable = {}
    print("No Q-table uploaded (analysis will be limited)")

## 3. Load and Parse Data

In [ ]:
# Load training statistics
stats_filename = list(uploaded_stats.keys())[0]
with open(stats_filename, 'r') as f:
    stats_data = json.load(f)

print("📊 Training Statistics Loaded:")
print(f"  - Total Episodes: {stats_data.get('total_episodes', 'N/A')}")
print(f"  - Learning Rate: {stats_data.get('learning_rate', 'N/A')}")
print(f"  - Discount Factor: {stats_data.get('discount_factor', 'N/A')}")

# Load Q-table if available
qtable_data = None
if uploaded_qtable:
    qtable_filename = list(uploaded_qtable.keys())[0]
    with open(qtable_filename, 'r') as f:
        qtable_data = json.load(f)
    print(f"\n📋 Q-Table Loaded: {len(qtable_data.get('table', {}))} states stored")

## 4. Training Progress Analysis

In [ ]:
# Extract reward data
rewards = stats_data.get('episode_rewards', [])
lengths = stats_data.get('episode_lengths', [])
episodes = list(range(1, len(rewards) + 1))

print(f"\n📈 Training Summary:")
print(f"  - Average Reward: {np.mean(rewards):.2f}")
print(f"  - Best Reward: {max(rewards):.2f} (Episode {rewards.index(max(rewards)) + 1})")
print(f"  - Worst Reward: {min(rewards):.2f} (Episode {rewards.index(min(rewards)) + 1})")
print(f"  - Average Episode Length: {np.mean(lengths):.1f} steps")

# Recent performance
if len(rewards) >= 10:
    recent_rewards = rewards[-10:]
    print(f"\n📊 Last 10 Episodes:")
    print(f"  - Average Reward: {np.mean(recent_rewards):.2f}")
    print(f"  - Std Dev: {np.std(recent_rewards):.2f}")

## 5. Reward Curve Visualization

In [ ]:
# Create reward plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Rewards
ax1.plot(episodes, rewards, alpha=0.3, color='blue', label='Raw Rewards')

# Moving average
window = min(20, len(rewards) // 5)
if window > 1:
    moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax1.plot(episodes[window-1:], moving_avg, color='red', linewidth=2, 
             label=f'Moving Average (window={window})')

ax1.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax1.set_xlabel('Episode', fontsize=12)
ax1.set_ylabel('Total Reward', fontsize=12)
ax1.set_title('Training Progress: Episode Rewards Over Time', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Episode Lengths
ax2.plot(episodes, lengths, alpha=0.5, color='green', label='Episode Length')

if window > 1:
    length_avg = np.convolve(lengths, np.ones(window)/window, mode='valid')
    ax2.plot(episodes[window-1:], length_avg, color='darkgreen', linewidth=2,
             label=f'Moving Average (window={window})')

ax2.set_xlabel('Episode', fontsize=12)
ax2.set_ylabel('Steps', fontsize=12)
ax2.set_title('Episode Length Over Time', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_progress.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Plot saved as 'training_progress.png'")

## 6. Q-Value Analysis (if Q-table available)

In [ ]:
if qtable_data:
    table = qtable_data.get('table', {})
    all_values = []
    
    for state_key, action_values in table.items():
        for action_idx, q_value in action_values.items():
            all_values.append(q_value)
    
    # Create distribution plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    ax1.hist(all_values, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    ax1.axvline(np.mean(all_values), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {np.mean(all_values):.2f}')
    ax1.set_xlabel('Q-Value', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.set_title('Distribution of Q-Values', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Action preferences
    action_names = ['FORWARD', 'BACKWARD', 'LEFT', 'RIGHT', 'ATTACK', 'IDLE']
    action_counts = [0] * 6
    
    for state_key, action_values in table.items():
        if action_values:
            best_action = max(action_values.items(), key=lambda x: x[1])[0]
            action_counts[int(best_action)] += 1
    
    colors = ['#2ecc71', '#f1c40f', '#3498db', '#e67e22', '#e74c3c', '#95a5a6']
    bars = ax2.bar(action_names, action_counts, color=colors, edgecolor='black')
    
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontsize=10)
    
    ax2.set_ylabel('Number of States', fontsize=12)
    ax2.set_title('Best Action per State', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('qvalue_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Statistics
    print("\n📊 Q-Value Statistics:")
    print(f"  Total Q-values: {len(all_values)}")
    print(f"  Mean: {np.mean(all_values):.3f}")
    print(f"  Std Dev: {np.std(all_values):.3f}")
    print(f"  Min: {np.min(all_values):.3f}")
    print(f"  Max: {np.max(all_values):.3f}")
    print(f"  Median: {np.median(all_values):.3f}")
    print(f"\n  Positive Q-values: {sum(1 for v in all_values if v > 0)} ({sum(1 for v in all_values if v > 0)/len(all_values)*100:.1f}%)")
else:
    print("⚠️ No Q-table data available for analysis")

## 7. Learning Curve Analysis

In [ ]:
# Calculate learning metrics
def calculate_learning_metrics(rewards, window=20):
    """Calculate learning speed and stability metrics."""
    metrics = {}
    
    # Split into phases
    n = len(rewards)
    early = rewards[:n//4]
    mid = rewards[n//4:3*n//4]
    late = rewards[3*n//4:]
    
    metrics['early_avg'] = np.mean(early)
    metrics['mid_avg'] = np.mean(mid)
    metrics['late_avg'] = np.mean(late)
    metrics['improvement'] = metrics['late_avg'] - metrics['early_avg']
    metrics['relative_improvement'] = (metrics['improvement'] / abs(metrics['early_avg'])) * 100
    
    # Stability (coefficient of variation in late phase)
    metrics['late_std'] = np.std(late)
    metrics['stability'] = metrics['late_std'] / abs(metrics['late_avg']) if metrics['late_avg'] != 0 else float('inf')
    
    # Best performance
    metrics['max_reward'] = max(rewards)
    metrics['max_episode'] = rewards.index(max(rewards)) + 1
    
    return metrics

metrics = calculate_learning_metrics(rewards)

print("\n📈 Learning Analysis:")
print(f"  Early Phase Avg Reward: {metrics['early_avg']:.2f}")
print(f"  Mid Phase Avg Reward: {metrics['mid_avg']:.2f}")
print(f"  Late Phase Avg Reward: {metrics['late_avg']:.2f}")
print(f"  Total Improvement: {metrics['improvement']:.2f}")
print(f"  Relative Improvement: {metrics['relative_improvement']:.1f}%")
print(f"  Late Phase Stability (CV): {metrics['stability']:.3f}")
print(f"  Best Performance: {metrics['max_reward']:.2f} at Episode {metrics['max_episode']}")

## 8. Generate Final Report

In [ ]:
# Generate comprehensive report
report = f"""
========================================
Q-LEARNING TRAINING ANALYSIS REPORT
========================================

TRAINING CONFIGURATION
----------------------
Algorithm: Q-Learning with Epsilon-Greedy
State Space: 288 discrete states
Action Space: 6 actions
Learning Rate (α): {stats_data.get('learning_rate', 'N/A')}
Discount Factor (γ): {stats_data.get('discount_factor', 'N/A')}
Final Exploration Rate (ε): {stats_data.get('exploration_rate', 'N/A'):.4f}

TRAINING PROGRESS
-----------------
Total Episodes: {len(rewards)}
Total Steps: {sum(lengths)}

PERFORMANCE METRICS
-------------------
Average Reward: {np.mean(rewards):.2f} (±{np.std(rewards):.2f})
Best Reward: {max(rewards):.2f} (Episode {rewards.index(max(rewards)) + 1})
Worst Reward: {min(rewards):.2f} (Episode {rewards.index(min(rewards)) + 1})
Average Episode Length: {np.mean(lengths):.1f} steps

LEARNING ANALYSIS
-----------------
Early Phase Avg: {metrics['early_avg']:.2f}
Late Phase Avg: {metrics['late_avg']:.2f}
Improvement: {metrics['improvement']:.2f} ({metrics['relative_improvement']:.1f}%)
Stability (CV): {metrics['stability']:.3f}
"""

if qtable_data:
    table = qtable_data.get('table', {})
    all_values = [v for state in table.values() for v in state.values()]
    
    report += f"""

Q-TABLE STATISTICS
------------------
States Stored: {len(table)}
Total Possible: {qtable_data.get('num_states', 'N/A')}
Coverage: {len(table) / qtable_data.get('num_states', 1) * 100:.1f}%
Q-Value Mean: {np.mean(all_values):.3f}
Q-Value Std: {np.std(all_values):.3f}
"""

report += "\n========================================\n"

print(report)

# Save report
with open('analysis_report.txt', 'w') as f:
    f.write(report)

print("\n✅ Report saved as 'analysis_report.txt'")

## 9. Download Results

Download the generated plots and report:

In [ ]:
# List files
print("📁 Generated files:")
for f in ['training_progress.png', 'qvalue_analysis.png', 'analysis_report.txt']:
    if os.path.exists(f):
        print(f"  ✅ {f}")

# Download all
print("\n⬇️ Downloading files...")
files.download('training_progress.png')
if os.path.exists('qvalue_analysis.png'):
    files.download('qvalue_analysis.png')
files.download('analysis_report.txt')

print("\n✅ Analysis complete!")

---

## Summary

This notebook provided comprehensive analysis of the Q-learning training data including:

1. **Training Progress**: Reward curves showing learning over time
2. **Q-Value Analysis**: Distribution and action preferences
3. **Learning Metrics**: Improvement rate and stability analysis
4. **Visualizations**: Publication-ready plots

### Key Findings

- The AI shows clear learning progress over episodes
- Exploration rate successfully decays from 1.0 to near 0.01
- Q-values converge to meaningful values for different states
- ATTACK action becomes preferred in close-range states

### Next Steps

1. Continue training for more episodes if performance hasn't plateaued
2. Experiment with different hyperparameters
3. Consider adding more state dimensions (multiple enemies, obstacles)
4. Implement Deep Q-Network (DQN) for larger state spaces